# ABT de Engajamento

Constrói uma Analytical Base Table de engajamento de alunos da plataforma ASN Rocks, combinando 5 dimensões comportamentais (frequência, progresso, profundidade, revisão e recência) em um score de classificação por aluno.

In [0]:
# Imports principais do PySpark
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.functions import expr, col, when

## Carga de Dados

Fontes: tabelas no catálogo `bronze.case_asn`

In [0]:
# Carga das tabelas-fonte do catálogo bronze.case_asn
progresso = spark.read.table("bronze.case_asn.progresso")            # progresso por curso/aluno
progresso_total = spark.read.table("bronze.case_asn.progresso_total")  # log aula-a-aula
aluno_geral = spark.read.table("bronze.case_asn.alunos_geral")        # cadastro e matrícula
tempo_aula = spark.read.table("bronze.case_asn.tempo_aula")           # duração oficial de cada aula

## 1. Dimensão: Frequência

Dias distintos em que o aluno acessou a plataforma (sessões >= 3 min).

In [0]:
# Frequência: dias distintos com acesso >= 3 min
base_frequencia = (
    progresso_total
    .withColumn("aluno_id", col("aluno_id").cast("int"))
    .withColumn(
        "ultima_visualizacao",
        expr("try_to_timestamp(ultima_visualizacao, \"yyyy-MM-dd'T'HH:mm:ssX\")")
    )
    # Extrai a data (sem hora) da última visualização
    .withColumn("data_visualizacao", f.to_date(col("ultima_visualizacao")))
    .withColumn(
        "tempo_segundos",
        expr("""
            coalesce(try_cast(get(split(tempo_aula, ':'), 0) as int), 0) * 3600 +
            coalesce(try_cast(get(split(tempo_aula, ':'), 1) as int), 0) * 60 +
            coalesce(try_cast(get(split(tempo_aula, ':'), 2) as int), 0)
        """).cast("int")
    )
    # Filtra sessões com >= 3 minutos de visualização
    .where(col("tempo_segundos") >= 180)
    .groupBy("aluno_id")
    # Conta dias distintos de acesso por aluno
    .agg(f.countDistinct("data_visualizacao").alias("Frequencia_Dias"))
)

display(base_frequencia)

## 2. Dimensão: Progresso

Progresso absoluto (%) e velocidade de progresso (%/dia) considerando apenas cursos ativos com carga > 10 min.

In [0]:
# Progresso: % absoluto e velocidade (%/dia) em cursos ativos com carga > 10 min
p = progresso.alias("p")
a = aluno_geral.alias("a")

# Base: join progresso x aluno, filtrando matrículas ativas e ignorando curso introdutório
base = (
    p.join(a, on=["matricula_id"], how="inner")
     .where(
        (f.col("a.status") == "Matrícula ativa") &
        (f.col("p.curso_titulo") != "Comece por aqui")
     )
)

# Denominador: cursos únicos por aluno + dias desde cadastro
denominador = (
    base
    .select("p.aluno_id", "p.curso_id", "a.cadastro_data")
    .dropDuplicates(["aluno_id", "curso_id"])
    .groupBy("p.aluno_id")
    .agg(
        f.countDistinct("p.curso_id").alias("qtd_cursos_ativos"),
        f.min("a.cadastro_data").alias("cadastro_data")
    )
    .withColumn(
        "dias_de_casa",
        f.greatest(f.datediff(f.current_date(), f.col("cadastro_data")), f.lit(1))
    )
)

# Numerador: soma do progresso em cursos com carga >= 10 min
numerador = (
    base
    .withColumn(
        "tempo_segundos",
        f.coalesce(f.split(f.col("p.carga_horaria"), ":").getItem(0).cast("int"), f.lit(0)) * 3600 +
        f.coalesce(f.split(f.col("p.carga_horaria"), ":").getItem(1).cast("int"), f.lit(0)) * 60 +
        f.coalesce(f.split(f.col("p.carga_horaria"), ":").getItem(2).cast("int"), f.lit(0))
    )
    .groupBy("p.aluno_id", "p.curso_id")
    .agg(
        f.max("tempo_segundos").alias("tempo_segundos"),
        f.max("p.progresso").alias("progresso")
    )
    .where(f.col("tempo_segundos") >= 600)
    .groupBy("aluno_id")
    .agg(f.sum("progresso").alias("soma_progresso"))
)

# Resultado: progresso absoluto (%) e velocidade (%/dia)
dm_progresso = (
    denominador.join(numerador, on="aluno_id", how="left")
    .fillna({"soma_progresso": 0})
    .withColumn(
        "progresso_plataforma_absoluto",
        (f.col("soma_progresso") / f.col("qtd_cursos_ativos"))
    )
    .withColumn(
        "velocidade_progresso",
        (f.col("progresso_plataforma_absoluto") / f.col("dias_de_casa"))
    )
    .drop("cadastro_data")
)

display(dm_progresso)

## 3. Dimensão: Profundidade

Tempo total assistido por aluno (segundos).

In [0]:
# Profundidade: tempo total assistido por aluno (segundos)
base_profundidade = (
    progresso_total
    .withColumn(
        "tempo_segundos",
        expr("""
            coalesce(try_cast(get(split(tempo_aula, ':'), 0) as int), 0) * 3600 +
            coalesce(try_cast(get(split(tempo_aula, ':'), 1) as int), 0) * 60 +
            coalesce(try_cast(get(split(tempo_aula, ':'), 2) as int), 0)
        """)
    )
    # Soma total de segundos por aluno
    .groupBy("aluno_id")
    .agg(f.sum("tempo_segundos").alias("profundidade_total_segundos"))
    .withColumn(
        "profundidade_total_minutos",
        f.col("profundidade_total_segundos") / 60
    )
)

display(base_profundidade)

## 4. Dimensão: Revisão

Quantidade de aulas onde o tempo assistido excede a duração oficial da aula.

In [0]:
# Revisão: aulas onde o tempo assistido excede a duração oficial
pt = progresso_total.alias("pt")
# Prepara tempo_aula: cast dos IDs para long e converte duração (timestamp) -> segundos
t = (
    tempo_aula
    .withColumn("curso_id", col("curso_id").cast("long"))
    .withColumn("modulo_id", col("modulo_id").cast("long"))
    .withColumn("aula_id", col("aula_id").cast("long"))
    .withColumn("tempo_aula_segundos", f.hour("Tempo") * 3600 + f.minute("Tempo") * 60 + f.second("Tempo"))
    .alias("t")
)

base_revisao = (
    pt.join(t, on=["curso_id", "modulo_id", "aula_id"], how="left")
    .withColumn(
        "tempo_assistido_segundos",
        expr("""
            coalesce(try_cast(get(split(tempo_aula, ':'), 0) as int), 0) * 3600 +
            coalesce(try_cast(get(split(tempo_aula, ':'), 1) as int), 0) * 60 +
            coalesce(try_cast(get(split(tempo_aula, ':'), 2) as int), 0)
        """)
    )
    # Filtra aulas onde assistiu mais do que a duração oficial
    .where(f.col("tempo_assistido_segundos") > f.col("tempo_aula_segundos"))
    .groupBy("aluno_id")
    # Conta aulas revisadas por aluno
    .agg(f.countDistinct(f.col("aula_id")).alias("qtd_aula_revistas"))
)

display(base_revisao)

## 5. Dimensão: Recência

Dias desde o último acesso do aluno à plataforma.

In [0]:
# Recência: dias desde o último acesso do aluno
p = progresso.alias("p")
a = aluno_geral.alias("a")

base_recencia = (
    p.join(a, on=["matricula_id"], how="inner")
    # Converte último_acesso (string ISO) -> timestamp
    .withColumn(
        "ultimo_acesso_ts",
        expr("try_to_timestamp(ultimo_acesso, \"yyyy-MM-dd'T'HH:mm:ssX\")")
    )
    # Se não houver último acesso, usa a data de cadastro como referência
    .withColumn(
        "data_comparacao",
        f.when(
            f.col("ultimo_acesso_ts").isNull(),
            f.col("a.cadastro_data")
        ).otherwise(f.col("ultimo_acesso_ts"))
    )
    # Calcula dias desde o acesso mais recente
    .groupBy("p.aluno_id")
    .agg(
        f.datediff(
            f.current_date(),
            f.max(f.to_date("data_comparacao"))
        ).alias("Dias_Recencia")
    )
)

display(base_recencia)

## 6. ABT Final

Junção das 5 dimensões por `aluno_id`, com classificação ordinal (Baixo/Médio/Alto) e score de engajamento ponderado.

**Pesos:** Recência 40%, Frequência 20%, Progresso 20%, Profundidade 10%, Revisão 10%.

In [0]:
# ABT Final: junção das 5 dimensões por aluno_id
al = aluno_geral.alias("al")
fr = base_frequencia.alias("fr")
pr = dm_progresso.alias("pr")
pf = base_profundidade.alias("pf")
rc = base_recencia.alias("rc")
rv = base_revisao.alias("rv")

# Join left: mantém todos os alunos mesmo sem dados em alguma dimensão
base_abt = (
    al.select("aluno_id").distinct()
    .join(fr, on=["aluno_id"], how="left")
    .join(pr, on=["aluno_id"], how="left")
    .join(pf, on=["aluno_id"], how="left")
    .join(rc, on=["aluno_id"], how="left")
    .join(rv, on=["aluno_id"], how="left")
    .select(
        f.col('aluno_id').alias("aluno"),
        f.col('Frequencia_Dias'),
        f.col('velocidade_progresso'),
        f.col('profundidade_total_segundos'),
        f.col('Dias_Recencia'),
        f.col('qtd_aula_revistas')
    )
    .fillna(0)
)

# Estatísticas (p25, mediana, p75, média, std) para classificação por percentis
estatistica = base_abt.select(
    f.percentile_approx("Frequencia_Dias", 0.25).alias("freq_p25"),
    f.percentile_approx("Frequencia_Dias", 0.5).alias("freq_mediana"),
    f.percentile_approx("Frequencia_Dias", 0.75).alias("freq_p75"),
    f.avg("Frequencia_Dias").alias("freq_media"),
    f.stddev("Frequencia_Dias").alias("freq_std"),
    f.percentile_approx("velocidade_progresso", 0.25).alias("prog_p25"),
    f.percentile_approx("velocidade_progresso", 0.5).alias("prog_mediana"),
    f.percentile_approx("velocidade_progresso", 0.75).alias("prog_p75"),
    f.avg("velocidade_progresso").alias("prog_media"),
    f.stddev("velocidade_progresso").alias("prog_std"),
    f.percentile_approx("profundidade_total_segundos", 0.25).alias("prof_p25"),
    f.percentile_approx("profundidade_total_segundos", 0.5).alias("prof_mediana"),
    f.percentile_approx("profundidade_total_segundos", 0.75).alias("prof_p75"),
    f.avg("profundidade_total_segundos").alias("prof_media"),
    f.stddev("profundidade_total_segundos").alias("prof_std"),
    f.percentile_approx("Dias_Recencia", 0.25).alias("rec_p25"),
    f.percentile_approx("Dias_Recencia", 0.5).alias("rec_mediana"),
    f.percentile_approx("Dias_Recencia", 0.75).alias("rec_p75"),
    f.avg("Dias_Recencia").alias("rec_media"),
    f.stddev("Dias_Recencia").alias("rec_std"),
    f.percentile_approx("qtd_aula_revistas", 0.25).alias("rev_p25"),
    f.percentile_approx("qtd_aula_revistas", 0.5).alias("rev_mediana"),
    f.percentile_approx("qtd_aula_revistas", 0.75).alias("rev_p75"),
    f.avg("qtd_aula_revistas").alias("rev_media"),
    f.stddev("qtd_aula_revistas").alias("rev_std")
).collect()[0]

# Classificação ordinal (Baixo/Médio/Alto) por percentis + padronização z-score + score ponderado
base_abt = (
    base_abt
    .withColumn(
        "frequencia",
        f.when(f.col('Frequencia_Dias') <= estatistica["freq_p25"], "Baixo")
         .when(f.col('Frequencia_Dias') >= estatistica["freq_p75"], "Alto")
         .otherwise("Medio")
    )
    .withColumn(
        "progresso",
        f.when(f.col('velocidade_progresso') <= estatistica["prog_p25"], "Baixo")
         .when(f.col('velocidade_progresso') >= estatistica["prog_p75"], "Alto")
         .otherwise("Medio")
    )
    .withColumn(
        "profundidade",
        f.when(f.col('profundidade_total_segundos') <= estatistica["prof_p25"], "Baixo")
         .when(f.col('profundidade_total_segundos') >= estatistica["prof_p75"], "Alto")
         .otherwise("Medio")
    )
    .withColumn(
        "recencia",
        f.when(f.col('Dias_Recencia') <= estatistica["rec_p25"], "Alto")
         .when(f.col('Dias_Recencia') >= estatistica["rec_p75"], "Baixo")
         .otherwise("Medio")
    )
    .withColumn(
        "revisao",
        f.when(f.col('qtd_aula_revistas') <= estatistica["rev_p25"], "Baixo")
         .when(f.col('qtd_aula_revistas') >= estatistica["rev_p75"], "Alto")
         .otherwise("Medio")
    )
    # Padronização z-score (recência invertida: menos dias = score maior)
    .withColumn("frequencia_pr", (f.col('Frequencia_Dias') - estatistica["freq_media"]) / estatistica["freq_std"])
    .withColumn("progresso_pr", (f.col('velocidade_progresso') - estatistica["prog_media"]) / estatistica["prog_std"])
    .withColumn("profundidade_pr", (f.col('profundidade_total_segundos') - estatistica["prof_media"]) / estatistica["prof_std"])
    .withColumn("recencia_pr", (estatistica["rec_media"] - f.col('Dias_Recencia')) / estatistica["rec_std"])
    .withColumn("revisao_pr", (f.col('qtd_aula_revistas') - estatistica["rev_media"]) / estatistica["rev_std"])
    # Score ponderado: Recência 40%, Frequência 20%, Progresso 20%, Profundidade 10%, Revisão 10%
    .withColumn("frequencia_peso", f.col('frequencia_pr') * 0.20)
    .withColumn("progresso_peso", f.col('progresso_pr') * 0.20)
    .withColumn("profundidade_peso", f.col('profundidade_pr') * 0.10)
    .withColumn("recencia_peso", f.col('recencia_pr') * 0.40)
    .withColumn("revisao_peso", f.col('revisao_pr') * 0.10)
    # Score final = soma dos pesos
    .withColumn(
        "score_classificao",
        f.col("frequencia_peso") + f.col("progresso_peso") + f.col("profundidade_peso") + f.col("recencia_peso") + f.col("revisao_peso")
    )
    # Classificação ordinal com regra de ouro: recência "Baixo" -> Alto Risco
    .withColumn(
        "classificao_ordinal",
        f.when(f.col("recencia") == "Baixo", "Alto Risco")
         .when(f.col("score_classificao") <= -1.5, "Alto Risco")
         .when((f.col("score_classificao") > -1.5) & (f.col("score_classificao") <= -0.5), "Em Risco")
         .when((f.col("score_classificao") > -0.5) & (f.col("score_classificao") <= 0.5), "Parcial")
         .when((f.col("score_classificao") > 0.5) & (f.col("score_classificao") <= 1.5), "Acima da Media")
         .when(f.col("score_classificao") > 1.5, "Engajado")
         .otherwise("Nao Classificado")
    )
)

# Seleção final das colunas de saída
base_abt_save = base_abt.select(
    'aluno', 'frequencia', 'progresso', 'profundidade', 'recencia', 'revisao',
    'score_classificao', 'classificao_ordinal'
)

display(base_abt_save)

In [0]:
# Persiste a ABT como tabela Delta no catálogo bronze.case_asn
base_abt_save.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.case_asn.abt_v1")

In [0]:
%sql 
select * from bronze.case_asn.abt_v1